# 02 · Fixed validation split

Related versions of one source text (human + its fusion/ai versions) must stay on the same side. There is no group id, so groups are recovered from the text:

1. inside every language × domain block, rare-word TF-IDF (`max_df`, sublinear tf) and the top-`neighbors` cosine neighbours of each text;
2. an edge for every neighbour pair with cosine ≥ `link_threshold` (the valley of the bimodal similarity histogram in `01_eda`);
3. an edge for texts sharing the same first `prefix_words` words when at most `prefix_max_docs` texts share it (template openings in parliament/court texts are shared by dozens of texts and are not linked);
4. connected components = groups; `StratifiedGroupKFold` (stratified by language|domain|label) holds out one fold of 10 % as validation.

The split is saved to `splits/split.csv` (`id`, `group`, `split`) and used by every notebook.

In [3]:
import pandas as pd

from hw3.config import load_config
from hw3.data import SPLIT_PATH, load_train
from hw3.split import build_split, leakage_audit, neighbors

REBUILD = False
cfg = load_config("split")
if REBUILD or not SPLIT_PATH.exists():
    build_split(cfg)
df = load_train()
cfg

{'seed': 42,
 'val_fraction': 0.1,
 'neighbors': 5,
 'min_df': 2,
 'max_df': 0.1,
 'link_threshold': 0.5,
 'prefix_words': 10,
 'prefix_max_docs': 4}

## Groups

In [4]:
sizes = df.groupby("group").size()
sizes.clip(upper=10).value_counts().sort_index().rename("groups (10 = 10+)").to_frame().T

,1,2,3,4,5,6,7,8,9,10
groups (10 = 10+),119586,37680,18378,2114,522,480,104,61,42,152


In [5]:
composition = df.groupby("group").label.agg(lambda s: "+".join(sorted(set(s))))
composition.value_counts().rename("groups").to_frame()

,groups
label,
ai,111890
fusion+human,51945
human,9596
fusion,3996
ai+human,848
ai+fusion+human,833
ai+fusion,11


## Balance of the split

In [6]:
counts = pd.crosstab([df.lang, df.domain, df.label], df.split)
counts.assign(val_share=(counts.val / counts.sum(1)).round(3))

/tmp/ipykernel_1614088/1083854396.py:2: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  counts.assign(val_share=(counts.val / counts.sum(1)).round(3))


split                                train   val  val_share
lang  domain                 label                         
de-DE bundestag              ai      17979  1998      0.100
                             fusion  16098  1789      0.100
                             human   13031  1448      0.100
      gutenberg              ai        324    36      0.100
                             fusion     35     2      0.054
                             human      30     1      0.032
en-EN arxiv_papers           ai      10845  1205      0.100
                             fusion   1650   183      0.100
                             human     366    41      0.101
      blog_authorship_corpus ai      14115  1569      0.100
                             fusion  11157  1240      0.100
                             human   11220  1246      0.100
      cnn_news               ai      17539  1949      0.100
                             fusion  10262  1140      0.100
                             human   12055  1340      0.100
      euro_court_cases       ai      10027  1114      0.100
                             fusion   9144  1016      0.100
                             human    5649   628      0.100
      gutenberg              ai       5723   636      0.100
                             fusion    307    34      0.100
                             human     194    22      0.102
      house_of_commons       ai      13768  1530      0.100
                             fusion   8647   961      0.100
                             human    6258   695      0.100
      student_essays         ai      19424  2159      0.100
                             fusion  18016  2001      0.100
                             human   11255  1251      0.100

## Leakage audit

`max_sim_to_train_p99`: 99th percentile of the highest cosine similarity of a validation text to any train text among its neighbours. `share_prefix_in_train`: validation texts whose 10-word opening appears in train (mostly templates); `share_rare_prefix_in_train`: the same restricted to openings shared by fewer than `prefix_max_docs` train texts.

In [7]:
nb = neighbors(cfg)
leakage_audit(df, nb, cfg)

,val_docs,max_sim_to_train_p99,share_sim_ge_threshold,share_prefix_in_train,share_rare_prefix_in_train
label,,,,,
ai,12196,0.4901,0.0,0.1370,0.0029
fusion,8366,0.4814,0.0,0.1450,0.0166
human,6672,0.4849,0.0,0.1302,0.0127


In [8]:
leakage_audit(df, nb, cfg, by="domain")

,val_docs,max_sim_to_train_p99,share_sim_ge_threshold,share_prefix_in_train,share_rare_prefix_in_train
domain,,,,,
arxiv_papers,1429,0.4969,0.0,0.0091,0.0021
blog_authorship_corpus,4055,0.4807,0.0,0.0027,0.0005
bundestag,5235,0.4374,0.0,0.3372,0.0256
cnn_news,4429,0.4908,0.0,0.0063,0.0023
euro_court_cases,2758,0.4871,0.0,0.3093,0.0174
gutenberg,731,0.4419,0.0,0.0192,0.0014
house_of_commons,3186,0.4839,0.0,0.2225,0.0094
student_essays,5411,0.4867,0.0,0.0665,0.0057


## Limits

- AI texts show no lexical link to a specific human text (their nearest neighbour is no closer than between unrelated texts of the same topic), so most of them are singleton groups; any topic overlap with their source can cross the split.
- Only the top `neighbors` neighbours are scanned and pairs below `link_threshold` are not linked, so heavily rewritten fusion texts can be missed.
- Linking is restricted to the same language × domain block.